In [17]:
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, inspect

import os


In [8]:
load_dotenv()

username = os.getenv("USERNAME")
password = os.getenv("PASSWORD")
host = os.getenv("HOST")
port = os.getenv("PORT")
db_name = os.getenv("DB_NAME")

conn_str = f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{db_name}"

In [9]:
engine = create_engine(conn_str)

In [10]:
# Reading data using a context manager
def read_sql_data(query):
    with engine.connect() as conn:
        df = pd.read_sql(text(query), conn)
    return df


In [11]:
def write_sql_data(df, table_name, mode="replace", index_allow=False):
    # Writing data using a transaction context manager (rolls back on failure)
    with engine.begin() as conn:
        df.to_sql(name=table_name, con=conn, if_exists=mode, index=index_allow)


In [19]:
def print_db_schema():
    """Inspects the database and prints a clean, formatted schema structure."""
    # Create the inspector instance from the engine
    inspector = inspect(engine)
    
    # Retrieve all table names in the default schema
    table_names = inspector.get_table_names()
    
    if not table_names:
        print("No tables found in the database.")
        return

    print(f"=== DATABASE SCHEMA OVERVIEW ({len(table_names)} Tables) ===")
    
    for table_name in table_names:
        print(f"\n📋 Table: {table_name}")
        print("-" * 40)
        
        # CORRECTED: Use get_pk_constraint and extract the list of columns
        pk_constraint = inspector.get_pk_constraint(table_name)
        pk_columns = pk_constraint.get('constrained_columns', [])
        
        fk_constraints = inspector.get_foreign_keys(table_name)
        
        # Map foreign keys to their local column names for easy lookup
        # Added safety check to ensure constrained_columns is not empty
        fk_map = {
            fk['constrained_columns'][0]: fk 
            for fk in fk_constraints 
            if fk.get('constrained_columns')
        }
        
        # Loop through columns and print details
        columns = inspector.get_columns(table_name)
        for col in columns:
            col_name = col['name']
            col_type = str(col['type'])
            is_nullable = "NULL" if col['nullable'] else "NOT NULL"
            
            # Identify Key tags
            key_tag = ""
            if col_name in pk_columns:
                key_tag = "🔑 [PK]"
            elif col_name in fk_map:
                target = fk_map[col_name]
                # Added safety check to ensure referred_columns exists and is not empty
                ref_col = target['referred_columns'][0] if target.get('referred_columns') else "unknown"
                key_tag = f"🔗 [FK -> {target['referred_table']}.{ref_col}]"
                
            print(f"  🔹 {col_name:<20} {col_type:<15} {is_nullable:<10} {key_tag}")
    
    print("\n" + "=" * 40)

# Usage Example:
# print_db_schema(engine)


In [24]:
# print_db_schema()

In [21]:
def print_db_schema_ui(include_tables=None, search_keyword=None):
    """
    Inspects the database and prints a clean, formatted schema structure.
    
    Parameters:
    - engine: SQLAlchemy engine instance.
    - include_tables (list, optional): Exact list of table names to display.
    - search_keyword (str, optional): Case-insensitive string to filter table names.
    """
    # Create the inspector instance from the engine
    inspector = inspect(engine)
    
    # Retrieve all table names in the default schema
    table_names = inspector.get_table_names()
    
    if not table_names:
        print("❌ No tables found in the database.")
        return

    # Apply table filters if provided
    if include_tables:
        table_names = [t for t in table_names if t in include_tables]
    elif search_keyword:
        table_names = [t for t in table_names if search_keyword.lower() in t.lower()]

    if not table_names:
        print("⚠️ No tables matched your filtering criteria.")
        return

    print(f"=== DATABASE SCHEMA OVERVIEW ({len(table_names)} Tables) ===")
    
    for table_name in table_names:
        print(f"\n📋 Table: {table_name}")
        print("-" * 55)
        
        # FIX: Correct method to get Primary Keys in SQLAlchemy
        pk_constraint = inspector.get_pk_constraint(table_name)
        pk_columns = pk_constraint.get('constrained_columns', [])
        
        # Get foreign key constraints
        fk_constraints = inspector.get_foreign_keys(table_name)
        
        # Map foreign keys to their local column names for easy lookup
        fk_map = {}
        for fk in fk_constraints:
            for local_col in fk.get('constrained_columns', []):
                fk_map[local_col] = fk
        
        # Loop through columns and print details
        columns = inspector.get_columns(table_name)
        for col in columns:
            col_name = col['name']
            col_type = str(col['type'])
            is_nullable = "NULL" if col['nullable'] else "NOT NULL"
            
            # Identify Key tags
            key_tag = ""
            if col_name in pk_columns:
                key_tag = "🔑 [PK]"
            elif col_name in fk_map:
                target = fk_map[col_name]
                ref_cols = ", ".join(target['referred_columns'])
                key_tag = f"🔗 [FK -> {target['referred_table']}({ref_cols})]"
                
            print(f"  🔹 {col_name:<20} {col_type:<15} {is_nullable:<10} {key_tag}")
    
    print("\n" + "=" * 55)


In [23]:
# print_db_schema_ui()

In [25]:
query = "SELECT * FROM customer;"

df = read_sql_data(query)

display(df.head(10))

,customer_id,store_id,first_name,last_name,email,address_id,activebool,create_date,last_update,active
0,524,1,Jared,Ely,jared.ely@sakilacustomer.org,530,True,2006-02-14,2013-05-26 14:49:45.738,1
1,1,1,Mary,Smith,mary.smith@sakilacustomer.org,5,True,2006-02-14,2013-05-26 14:49:45.738,1
2,2,1,Patricia,Johnson,patricia.johnson@sakilacustomer.org,6,True,2006-02-14,2013-05-26 14:49:45.738,1
3,3,1,Linda,Williams,linda.williams@sakilacustomer.org,7,True,2006-02-14,2013-05-26 14:49:45.738,1
4,4,2,Barbara,Jones,barbara.jones@sakilacustomer.org,8,True,2006-02-14,2013-05-26 14:49:45.738,1
5,5,1,Elizabeth,Brown,elizabeth.brown@sakilacustomer.org,9,True,2006-02-14,2013-05-26 14:49:45.738,1
6,6,2,Jennifer,Davis,jennifer.davis@sakilacustomer.org,10,True,2006-02-14,2013-05-26 14:49:45.738,1
7,7,1,Maria,Miller,maria.miller@sakilacustomer.org,11,True,2006-02-14,2013-05-26 14:49:45.738,1
8,8,2,Susan,Wilson,susan.wilson@sakilacustomer.org,12,True,2006-02-14,2013-05-26 14:49:45.738,1
9,9,2,Margaret,Moore,margaret.moore@sakilacustomer.org,13,True,2006-02-14,2013-05-26 14:49:45.738,1
